In [ ]:
import os
import pandas as pd
import fitz
from collections import defaultdict
import shutil
import re  # <<< added for targeted pattern handling

# Configuration
input_folder = input("Enter the path to where your PDF files are kept that you want to email to customers: ")
base_output_folder = os.path.join(input_folder, "Grouped_PDFs")
os.makedirs(base_output_folder, exist_ok=True)

# >>> NEW: Special folder for PDFs with blank email
no_email_folder = os.path.join(base_output_folder, "NoEmailFound")
os.makedirs(no_email_folder, exist_ok=True)

do_not_email_folder = os.path.join(base_output_folder, "DoNotEmail")
os.makedirs(do_not_email_folder, exist_ok=True)

# Store all PDFs' structured data
all_dfs = []

def extract_structured_text(pdf_path):
    doc = fitz.open(pdf_path)
    all_blocks = []

    for page_num, page in enumerate(doc):
        blocks = page.get_text("dict")["blocks"]
        for b in blocks:
            if "lines" in b:
                for line in b["lines"]:
                    for span in line["spans"]:
                        bbox = tuple(round(coord) for coord in span["bbox"])
                        all_blocks.append({
                            "page": page_num + 1,
                            "text": span["text"],
                            "x0": bbox[0],
                            "y0": bbox[1],
                            "x1": bbox[2],
                            "y1": bbox[3]
                        })

    df = pd.DataFrame(all_blocks)
    return df.sort_values(by=["page", "y0", "x0"])

def group_by_dynamic_columns(df, y_tolerance=2, x_bin_width=20):
    df = df.sort_values(by=["page", "y0", "x0"]).reset_index(drop=True)
    structured_rows = []

    for page in df["page"].unique():
        page_df = df[df["page"] == page].copy()
        rows = []

        for _, row in page_df.iterrows():
            y0 = row["y0"]
            matched = False

            for r in rows:
                if abs(r["y0"] - y0) <= y_tolerance:
                    r["cells"].append(row)
                    matched = True
                    break

            if not matched:
                rows.append({"y0": y0, "cells": [row]})

        for r in rows:
            bins = defaultdict(list)
            for cell in r["cells"]:
                bin_index = int(cell["x0"] // x_bin_width)
                bins[bin_index].append(cell["text"])

            sorted_bins = [bins[i] for i in sorted(bins)]
            structured_row = [" ".join(bin) for bin in sorted_bins]
            structured_rows.append(structured_row)

    return pd.DataFrame(structured_rows)

# ---------------- PROCESS ALL PDFs ----------------
for filename in os.listdir(input_folder):
    if filename.lower().endswith(".pdf"):
        pdf_path = os.path.join(input_folder, filename)

        df = extract_structured_text(pdf_path)
        df2 = group_by_dynamic_columns(df)

        df2["source_pdf"] = filename
        all_dfs.append(df2)

final_df = pd.concat(all_dfs, ignore_index=True)

# Extract Email rows + next 2 rows
keyword = "Email"
column_name = 0
matches = final_df.index[final_df[column_name].astype(str).str.strip() == keyword]

rows_to_include = []
for idx in matches:
    end_idx = idx + 3
    rows_to_include.extend(range(idx, min(end_idx, len(final_df))))

rows_to_include = sorted(set(rows_to_include))
df_filtered = final_df.iloc[rows_to_include].reset_index(drop=True)
df_filtered = df_filtered[(df_filtered[0] != 'Customer Phone') & (df_filtered[0] != 'Customer Fax')]

# Combine columns 0–7 into one column called 'Email'
def combine_email_values(row):
    values = [str(v).strip() for v in row[:8] if str(v).strip() not in ["Email", "None", "nan", ""]]
    return ", ".join(values)

df_filtered["Email"] = df_filtered.apply(combine_email_values, axis=1)
df_filtered = df_filtered[["source_pdf", "Email"]]
df_filtered = df_filtered.groupby('source_pdf')['Email'].agg(lambda x: ''.join(x)).reset_index()
df_filtered['Email'] = df_filtered['Email'].str.replace(r'\*+', '', regex=True).str.strip()

# >>> NEW: Targeted move of only specific notes to Comment, and strip them from Email
def extract_comments_and_clean(email_str):
    """
    Moves ONLY the following to comments if present anywhere in the string:
      - 'US Mail' or 'USMail'
      - 'read receipt' (with or without parentheses)
      - '& PM' or '&PM'
    Returns (cleaned_email, comments_str).
    """
    s = email_str or ""
    comments = []

    # Detect presence (case-insensitive) for standardized labels
    if re.search(r'\bUS\s*Mail\b', s, flags=re.IGNORECASE):
        comments.append("US Mail")
    if re.search(r'\bUSMail\b', s, flags=re.IGNORECASE):
        comments.append("USMail")
    if re.search(r'\bread\s+receipt\b', s, flags=re.IGNORECASE):
        comments.append("read receipt")
    if re.search(r'&\s*PM\b', s):
        comments.append("PM")

    # Remove only those occurrences (including common delimiters around them)
    # Slash before US Mail/USMail (e.g., '/US Mail', '/USMail')
    s = re.sub(r'(?i)\s*/\s*US\s*Mail', '', s)
    s = re.sub(r'(?i)\s*/\s*USMail', '', s)

    # 'read receipt' with or without parentheses
    s = re.sub(r'(?i)\s*\(\s*read\s+receipt\s*\)\s*', '', s)  # (read receipt)
    s = re.sub(r'(?i)\s*read\s+receipt\s*', '', s)            # read receipt

    # '& PM' or '&PM'
    s = re.sub(r'\s*&\s*PM\b', '', s)

    # Tidy spaces and commas after removals
    s = re.sub(r'\s+', ' ', s)
    s = re.sub(r'\s*,\s*', ', ', s)
    s = re.sub(r'(,\s*){2,}', ', ', s)
    s = s.strip(' ,')

    # De-duplicate comments preserving order
    seen = set()
    comments_unique = []
    for c in comments:
        if c not in seen:
            comments_unique.append(c)
            seen.add(c)

    return s, '; '.join(comments_unique)

# Apply extraction
clean_cols = df_filtered['Email'].apply(extract_comments_and_clean)
df_filtered['Email'] = clean_cols.apply(lambda x: x[0])     # cleaned email string
df_filtered['Comment'] = clean_cols.apply(lambda x: x[1])   # ONLY targeted notes
df_filtered['Email'] = df_filtered['Email'].str.replace(r'(?i)/\s*cc\s*([A-Za-z0-9._%+-]+)', r';\1', regex=True)
df_filtered['Email'] = df_filtered['Email'].str.replace(r'(?i)/\s*cc\b', r';', regex=True)
df_filtered['Email'] = df_filtered['Email'].str.replace(r'\s*/cc\s*', ';', flags=re.IGNORECASE, regex=True)
df_filtered['Email'] = df_filtered['Email'].str.replace(r'/', ';', regex=True)


def normalize_domains(email_str):
    """
    Normalize ';' separated tokens by using the domain from the FIRST email:
      - If a token contains '@', replace everything from '@' onward with the first domain.
      - If a token has no '@' but is a valid local-part (letters/digits/._%+-), append the first domain.
      - Non-email tokens are left untouched.
    Also normalizes commas to semicolons and tidies separators.
    """
    if not isinstance(email_str, str) or not email_str.strip():
        return email_str

    # Unify commas to semicolons and trim
    s = re.sub(r'\s*,\s*', ';', email_str.strip())

    # Split on semicolons
    parts = [p.strip() for p in s.split(';') if p.strip()]
    if not parts:
        return email_str

    # Find domain from the first email-like token that contains '@'
    domain = None
    for p in parts:
        if '@' in p:
            at_idx = p.find('@')
            dom = p[at_idx:]  # includes '@'
            if len(dom) > 1:
                domain = dom
                break

    # If no domain found, do nothing
    if not domain:
        out = ';'.join(parts)
        out = re.sub(r'\s*;\s*', ';', out)
        out = re.sub(r';{2,}', ';', out)
        return out.strip(' ;,')

    normalized = []
    for i, p in enumerate(parts):
        if i == 0:
            # Keep the first token as-is (it carries the canonical domain)
            normalized.append(p)
            continue

        token = p.strip(' ,')

        if '@' in token:
            # Replace everything from '@' with the first domain
            local = token.split('@', 1)[0]
            normalized.append(local + domain)
        else:
            # Append domain if token is a valid local-part (no '@' present)
            # Accept letters/digits and . _ % + - ; no need to require a dot
            if re.match(r'^[A-Za-z0-9._%+-]+$', token):
                normalized.append(token + domain)
            else:
                normalized.append(token)  # leave non-email tokens

    out = ';'.join(normalized)
    out = re.sub(r'\s*;\s*', ';', out)
    out = re.sub(r';{2,}', ';', out)
    return out.strip(' ;,')


# >>> NEW: Normalize domains across ';' separated tokens using the first email's domain
df_filtered['Email'] = df_filtered['Email'].apply(normalize_domains)


# >>> NEW: Remove duplicate emails (preserve order)
def dedupe_semicolon_list(s):
    if not isinstance(s, str) or not s.strip():
        return s
    parts = [p.strip() for p in s.split(';') if p.strip()]
    seen = set()
    out = []
    for p in parts:
        if p not in seen:
            out.append(p)
            seen.add(p)
    return ';'.join(out)

df_filtered['Email'] = df_filtered['Email'].apply(dedupe_semicolon_list)


# ---------------- GROUP PDFs BY EMAIL ----------------
for _, row in df_filtered.iterrows():
    pdf_name = row['source_pdf']
    # Split emails by comma and drop empties after stripping
    emails = [e.strip() for e in str(row['Email']).split(",") if e.strip()]

    src_pdf_path = os.path.join(input_folder, pdf_name)

    # >>> Revised: If NO '@' appears anywhere in the cleaned Email string, move to NoEmailFound (keep original filename)
    emails_with_at = [e for e in emails if '@' in e]
    if not emails_with_at:
        dest_pdf_path = os.path.join(no_email_folder, pdf_name)
        if not os.path.exists(dest_pdf_path):  # Avoid duplicate copies
            shutil.copy(src_pdf_path, dest_pdf_path)
        continue

    # Existing behavior: use cleaned emails (with '@') for folder grouping
    for email in emails_with_at:
        email_folder = os.path.join(base_output_folder, email)
        os.makedirs(email_folder, exist_ok=True)

        dest_pdf_path = os.path.join(email_folder, pdf_name)
        if not os.path.exists(dest_pdf_path):  # Avoid duplicate copies
            shutil.copy(src_pdf_path, dest_pdf_path)

excel_path = os.path.join(base_output_folder, "Email_and_PDF_Mapping.xlsx")
df_filtered_sorted = df_filtered.sort_values(by="Email")
df_filtered_sorted.to_excel(excel_path, index=False)

print("✅ PDFs have been grouped by email into folders inside:", base_output_folder)
print(f"📂 PDFs with blank Email were placed in: {no_email_folder}")

✅ PDFs have been grouped by email into folders inside: C:\Users\IlaBarshilia\Downloads\Grouping PDFs by Email\Grouped_PDFs
📂 PDFs with blank Email were placed in: C:\Users\IlaBarshilia\Downloads\Grouping PDFs by Email\Grouped_PDFs\NoEmailFound


##### Version 2 08Dec2025 - Added DoNotEmail folder for Payapp and Adj Inv cases and added clause for Email found in columns 1 or 2 instead of just first column

In [140]:
import os
import pandas as pd
import fitz
from collections import defaultdict
import shutil
import re  # <<< added for targeted pattern handling
import numpy as np

# Configuration
input_folder = input("Enter the path to where your PDF files are kept that you want to email to customers: ")
base_output_folder = os.path.join(input_folder, "Grouped_PDFs")
os.makedirs(base_output_folder, exist_ok=True)

# >>> NEW: Special folder for PDFs with blank email
no_email_folder = os.path.join(base_output_folder, "NoEmailFound")
os.makedirs(no_email_folder, exist_ok=True)

do_not_email_folder = os.path.join(base_output_folder, "DoNotEmail")
os.makedirs(do_not_email_folder, exist_ok=True)

# Store all PDFs' structured data
all_dfs = []

def extract_structured_text(pdf_path):
    doc = fitz.open(pdf_path)
    all_blocks = []

    for page_num, page in enumerate(doc):
        blocks = page.get_text("dict")["blocks"]
        for b in blocks:
            if "lines" in b:
                for line in b["lines"]:
                    for span in line["spans"]:
                        bbox = tuple(round(coord) for coord in span["bbox"])
                        all_blocks.append({
                            "page": page_num + 1,
                            "text": span["text"],
                            "x0": bbox[0],
                            "y0": bbox[1],
                            "x1": bbox[2],
                            "y1": bbox[3]
                        })

    df = pd.DataFrame(all_blocks)
    return df.sort_values(by=["page", "y0", "x0"])

def group_by_dynamic_columns(df, y_tolerance=2, x_bin_width=20):
    df = df.sort_values(by=["page", "y0", "x0"]).reset_index(drop=True)
    structured_rows = []

    for page in df["page"].unique():
        page_df = df[df["page"] == page].copy()
        rows = []

        for _, row in page_df.iterrows():
            y0 = row["y0"]
            matched = False

            for r in rows:
                if abs(r["y0"] - y0) <= y_tolerance:
                    r["cells"].append(row)
                    matched = True
                    break

            if not matched:
                rows.append({"y0": y0, "cells": [row]})

        for r in rows:
            bins = defaultdict(list)
            for cell in r["cells"]:
                bin_index = int(cell["x0"] // x_bin_width)
                bins[bin_index].append(cell["text"])

            sorted_bins = [bins[i] for i in sorted(bins)]
            structured_row = [" ".join(bin) for bin in sorted_bins]
            structured_rows.append(structured_row)

    return pd.DataFrame(structured_rows)

# ---------------- PROCESS ALL PDFs ----------------
for filename in os.listdir(input_folder):
    if filename.lower().endswith(".pdf"):
        pdf_path = os.path.join(input_folder, filename)

        df = extract_structured_text(pdf_path)
        df2 = group_by_dynamic_columns(df)

        df2["source_pdf"] = filename
        all_dfs.append(df2)

final_df = pd.concat(all_dfs, ignore_index=True)

In [142]:
final_df=final_df[final_df['source_pdf']=='3-D Paving & Sealcoating - Inv 766469, 11-30-25.pdf']

In [143]:
def extract_right_concat(
    df: pd.DataFrame,
    keywords=("Job No", "P.O. #"),
    in_column: str | None = None,
    sep: str = " ",
    case_insensitive: bool = True
) -> pd.Series:

    cols = list(df.columns)
    second_last_idx = len(cols) - 2  # index of second-last column

    # Precompute lowercase keywords if needed
    kw_set = {k.lower() if case_insensitive else k for k in keywords}

    def _match(cell) -> bool:
        if cell is None or (isinstance(cell, float) and np.isnan(cell)):
            return False
        s = str(cell)
        s_cmp = s.lower() if case_insensitive else s
        return any(k in s_cmp for k in kw_set)

    def _concat_row(row: pd.Series) -> str | float:
        # Find keyword position
        if in_column is not None:
            if in_column not in df.columns:
                raise ValueError(f"`in_column` '{in_column}' not in DataFrame columns.")
            pos = cols.index(in_column)
            if not _match(row[in_column]):
                return np.nan
        else:
            # Search across the entire row for the first occurrence
            pos = None
            for j, c in enumerate(cols):
                if _match(row[c]):
                    pos = j
                    break
            if pos is None:
                return np.nan

        # Compute slice start (immediately to the right) and end (second-last col inclusive)
        start = pos + 1
        end = second_last_idx
        if start > end:
            return np.nan  # nothing to extract

        # Extract, drop None/NaN/empty, concatenate
        vals = row.iloc[start:end+1].tolist()
        cleaned = []
        for v in vals:
            if v is None or (isinstance(v, float) and np.isnan(v)):
                continue
            s = str(v).strip()
            if s:  # exclude empty after trimming
                cleaned.append(s)
        if not cleaned:
            return np.nan
        return sep.join(cleaned)

    return df.apply(_concat_row, axis=1)
final_df["DoNotEmail"] = extract_right_concat(final_df, keywords=("Job No", "P.O. #"))

In [148]:
final_df2 = final_df[['source_pdf', 'DoNotEmail']]
final_df2 = final_df2[final_df2["DoNotEmail"].str.contains('Pay App|Adj Inv', case=False, na=False)]
final_df2.drop_duplicates(subset=['source_pdf'], keep='first')

# Work on a copy with a fresh RangeIndex so iloc works with your window logic
fd = final_df.reset_index(drop=True)

keyword = "Email"
columns_to_check = [0, 1, 2]

# Find rows where any of these columns contains 'Email' (case-insensitive)
matches = fd.index[
    fd[columns_to_check]
    .astype(str)
    .apply(lambda row: any(keyword.lower() in str(val).lower() for val in row), axis=1)]

# Include Email row + next 2 rows
rows_to_include = []
for pos in matches:  # pos are now positions 0..len(fd)-1
    end_pos = min(pos + 3, len(fd))
    rows_to_include.extend(range(pos, end_pos))

rows_to_include = sorted(set(rows_to_include))
df_filtered = fd.iloc[rows_to_include].reset_index(drop=True)

# Remove unwanted rows
df_filtered[0] = df_filtered[0].astype(str).str.replace("\u00A0", " ").str.strip()
df_filtered = df_filtered[(df_filtered[0] != 'Customer Phone') & (df_filtered[0] != 'Customer Fax')]
df_filtered.drop(columns="DoNotEmail", inplace=True)

In [ ]:
def combine_email_per_pdf_blocks(df_filtered: pd.DataFrame) -> pd.DataFrame:
    if 'source_pdf' not in df_filtered.columns:
        raise ValueError("Expected 'source_pdf' column in df_filtered.")
    src_idx = df_filtered.columns.get_loc('source_pdf')
    end_col = src_idx - 1  # last usable column index

    def norm(x):
        if x is None or (isinstance(x, float) and pd.isna(x)):
            return ""
        return str(x).replace("\u00A0", " ").strip()

    search_cols = [0, 1, 2]
    markers = {"email", "none", "nan", ""}

    out_rows = []
    for pdf, grp in df_filtered.groupby('source_pdf', sort=False):
        email_pos = None
        first_match_row = None

        # Find first occurrence of 'Email'
        for ridx, row in grp.iterrows():
            for c in search_cols:
                if c > end_col or c >= len(row):
                    continue
                if "email" in norm(row.iloc[c]).lower():
                    email_pos = c
                    first_match_row = ridx
                    break
            if email_pos is not None:
                break

        if email_pos is None or end_col < 0:
            out_rows.append({"source_pdf": pdf, "Email": ""})
            continue

        pieces = []
        for ridx, row in grp.iterrows():
            start_col = (email_pos + 1) if ridx == first_match_row else 0
            start_col = max(0, min(start_col, end_col))
            for col_idx in range(start_col, end_col + 1):
                val = norm(row.iloc[col_idx])
                if not val or val.lower() in markers:
                    continue
                pieces.append(val)

        combined = "".join(pieces)
        out_rows.append({"source_pdf": pdf, "Email": combined})

    return pd.DataFrame(out_rows, columns=["source_pdf", "Email"])

df_email_by_pdf = combine_email_per_pdf_blocks(df_filtered)

# Replace df_filtered for downstream steps
df_filtered = df_email_by_pdf[['source_pdf', 'Email']]

df_filtered = df_filtered.groupby('source_pdf')['Email'].agg(lambda x: ''.join(x)).reset_index()
df_filtered['Email'] = df_filtered['Email'].str.replace(r'\*+', '', regex=True).str.strip()

# >>> NEW: Targeted move of only specific notes to Comment, and strip them from Email
def extract_comments_and_clean(email_str):
    s = email_str or ""
    comments = []

    # Detect presence (case-insensitive) for standardized labels
    if re.search(r'\bUS\s*Mail\b', s, flags=re.IGNORECASE):
        comments.append("US Mail")
    if re.search(r'\bUSMail\b', s, flags=re.IGNORECASE):
        comments.append("USMail")
    if re.search(r'\bread\s+receipt\b', s, flags=re.IGNORECASE):
        comments.append("read receipt")
    if re.search(r'&\s*PM\b', s):
        comments.append("PM")

    # Remove only those occurrences (including common delimiters around them)
    # Slash before US Mail/USMail (e.g., '/US Mail', '/USMail')
    s = re.sub(r'(?i)\s*/\s*US\s*Mail', '', s)
    s = re.sub(r'(?i)\s*/\s*USMail', '', s)

    # 'read receipt' with or without parentheses
    s = re.sub(r'(?i)\s*\(\s*read\s+receipt\s*\)\s*', '', s)  # (read receipt)
    s = re.sub(r'(?i)\s*read\s+receipt\s*', '', s)            # read receipt

    # '& PM' or '&PM'
    s = re.sub(r'\s*&\s*PM\b', '', s)

    # Tidy spaces and commas after removals
    s = re.sub(r'\s+', ' ', s)
    s = re.sub(r'\s*,\s*', ', ', s)
    s = re.sub(r'(,\s*){2,}', ', ', s)
    s = s.strip(' ,')

    # De-duplicate comments preserving order
    seen = set()
    comments_unique = []
    for c in comments:
        if c not in seen:
            comments_unique.append(c)
            seen.add(c)

    return s, '; '.join(comments_unique)

# Apply extraction
clean_cols = df_filtered['Email'].apply(extract_comments_and_clean)
df_filtered['Email'] = clean_cols.apply(lambda x: x[0])     # cleaned email string
df_filtered['Comment'] = clean_cols.apply(lambda x: x[1])   # ONLY targeted notes
df_filtered['Email'] = df_filtered['Email'].str.replace(r'(?i)/\s*cc\s*([A-Za-z0-9._%+-]+)', r';\1', regex=True)
df_filtered['Email'] = df_filtered['Email'].str.replace(r'(?i)/\s*cc\b', r';', regex=True)
df_filtered['Email'] = df_filtered['Email'].str.replace(r'\s*/cc\s*', ';', flags=re.IGNORECASE, regex=True)
df_filtered['Email'] = df_filtered['Email'].str.replace(r'/', ';', regex=True)


def normalize_domains(email_str):
    """
    Normalize ';' separated tokens by using the domain from the FIRST email:
      - If a token contains '@', replace everything from '@' onward with the first domain.
      - If a token has no '@' but is a valid local-part (letters/digits/._%+-), append the first domain.
      - Non-email tokens are left untouched.
    Also normalizes commas to semicolons and tidies separators.
    """
    if not isinstance(email_str, str) or not email_str.strip():
        return email_str

    # Unify commas to semicolons and trim
    s = re.sub(r'\s*,\s*', ';', email_str.strip())

    # Split on semicolons
    parts = [p.strip() for p in s.split(';') if p.strip()]
    if not parts:
        return email_str

    # Find domain from the first email-like token that contains '@'
    domain = None
    for p in parts:
        if '@' in p:
            at_idx = p.find('@')
            dom = p[at_idx:]  # includes '@'
            if len(dom) > 1:
                domain = dom
                break

    # If no domain found, do nothing
    if not domain:
        out = ';'.join(parts)
        out = re.sub(r'\s*;\s*', ';', out)
        out = re.sub(r';{2,}', ';', out)
        return out.strip(' ;,')

    normalized = []
    for i, p in enumerate(parts):
        if i == 0:
            # Keep the first token as-is (it carries the canonical domain)
            normalized.append(p)
            continue

        token = p.strip(' ,')

        if '@' in token:
            # Replace everything from '@' with the first domain
            local = token.split('@', 1)[0]
            normalized.append(local + domain)
        else:
            # Append domain if token is a valid local-part (no '@' present)
            # Accept letters/digits and . _ % + - ; no need to require a dot
            if re.match(r'^[A-Za-z0-9._%+-]+$', token):
                normalized.append(token + domain)
            else:
                normalized.append(token)  # leave non-email tokens

    out = ';'.join(normalized)
    out = re.sub(r'\s*;\s*', ';', out)
    out = re.sub(r';{2,}', ';', out)
    return out.strip(' ;,')


# >>> NEW: Normalize domains across ';' separated tokens using the first email's domain
df_filtered['Email'] = df_filtered['Email'].apply(normalize_domains)
df_filtered = df_filtered.merge(final_df2, on='source_pdf', how='left')


def dedupe_semicolon_list(s):
    if not isinstance(s, str) or not s.strip():
        return s
    parts = [p.strip() for p in s.split(';') if p.strip()]
    seen = set()
    out = []
    for p in parts:
        if p not in seen:
            out.append(p)
            seen.add(p)
    return ';'.join(out)

df_filtered['Email'] = df_filtered['Email'].apply(dedupe_semicolon_list)

# Ensure folders exist (if not already created above)
os.makedirs(base_output_folder, exist_ok=True)
os.makedirs(no_email_folder, exist_ok=True)
os.makedirs(do_not_email_folder, exist_ok=True)  # <<< make sure this exists

# ---------------- GROUP PDFs BY EMAIL (with DoNotEmail precedence) ----------------
for _, row in df_filtered.iterrows():
    pdf_name = row['source_pdf']
    src_pdf_path = os.path.join(input_folder, pdf_name)

    # 1) >>> PRIORITY RULE: If DoNotEmail is present (non-NaN/non-empty), put PDF in DoNotEmail folder and skip further logic
    dni_val = row.get('DoNotEmail', None)
    dni_has_value = isinstance(dni_val, str) and dni_val.strip() != ""
    if (dni_val is not None) and (not isinstance(dni_val, float)) and dni_has_value:
        dest_pdf_path = os.path.join(do_not_email_folder, pdf_name)
        if not os.path.exists(dest_pdf_path):  # Avoid duplicate copies
            shutil.copy(src_pdf_path, dest_pdf_path)
        # Skip to next PDF; DoNotEmail takes precedence over NoEmailFound and email grouping
        continue

    # 2) Existing behavior: split Email tokens and determine grouping
    # Normalize split on semicolon OR comma and drop empties
    email_raw = row.get('Email', "")
    emails = [e.strip() for e in re.split(r'[;,]', str(email_raw)) if e.strip()]

    # If NO '@' appears anywhere in the cleaned Email string, move to NoEmailFound (keep original filename)
    emails_with_at = [e for e in emails if '@' in e]
    if not emails_with_at:
        dest_pdf_path = os.path.join(no_email_folder, pdf_name)
        if not os.path.exists(dest_pdf_path):  # Avoid duplicate copies
            shutil.copy(src_pdf_path, dest_pdf_path)
        continue

    # Use cleaned emails (with '@') for folder grouping
    for email in emails_with_at:
        email_folder = os.path.join(base_output_folder, email)
        os.makedirs(email_folder, exist_ok=True)

        dest_pdf_path = os.path.join(email_folder, pdf_name)
        if not os.path.exists(dest_pdf_path):  # Avoid duplicate copies
            shutil.copy(src_pdf_path, dest_pdf_path)

# Write mapping
excel_path = os.path.join(base_output_folder, "Email_and_PDF_Mapping.xlsx")
df_filtered_sorted = df_filtered.sort_values(by="Email")
df_filtered_sorted.to_excel(excel_path, index=False)

print("✅ PDFs have been grouped by email into folders inside:", base_output_folder)
print(f"🚫 PDFs flagged with DoNotEmail were placed in: {do_not_email_folder}")
print(f"📂 PDFs with blank Email were placed in: {no_email_folder}")


NotADirectoryError: [WinError 267] The directory name is invalid: 'C:\\Users\\IlaBarshilia\\Downloads\\split_invoices_of_Nov CH Invoices MIA #1_2025-Dec-04\\Grouped_PDFs\\erin@3-dpaving.comCustomer Phone954-933-2053Customer Fax954-933-1380REF:2510 W Broward Blvd'

In [136]:

#!/usr/bin/env python3
import sys
import traceback

# ---------------- YOUR EXISTING IMPORTS ----------------
# import os, re, shutil, pandas as pd, numpy as np
# from your_module import extract_right_concat, normalize_domains, dedupe_semicolon_list, ...
# ------------------------------------------------------

def main():
    import os
    import pandas as pd
    import fitz
    from collections import defaultdict
    import shutil
    import re  # <<< added for targeted pattern handling
    import numpy as np

    # Configuration
    input_folder = input("Enter the path to where your PDF files are kept that you want to email to customers: ")
    base_output_folder = os.path.join(input_folder, "Grouped_PDFs")
    os.makedirs(base_output_folder, exist_ok=True)

    # >>> NEW: Special folder for PDFs with blank email
    no_email_folder = os.path.join(base_output_folder, "NoEmailFound")
    os.makedirs(no_email_folder, exist_ok=True)

    do_not_email_folder = os.path.join(base_output_folder, "DoNotEmail")
    os.makedirs(do_not_email_folder, exist_ok=True)

    # Store all PDFs' structured data
    all_dfs = []

    def extract_structured_text(pdf_path):
        doc = fitz.open(pdf_path)
        all_blocks = []

        for page_num, page in enumerate(doc):
            blocks = page.get_text("dict")["blocks"]
            for b in blocks:
                if "lines" in b:
                    for line in b["lines"]:
                        for span in line["spans"]:
                            bbox = tuple(round(coord) for coord in span["bbox"])
                            all_blocks.append({
                                "page": page_num + 1,
                                "text": span["text"],
                                "x0": bbox[0],
                                "y0": bbox[1],
                                "x1": bbox[2],
                                "y1": bbox[3]
                            })

        df = pd.DataFrame(all_blocks)
        return df.sort_values(by=["page", "y0", "x0"])

    def group_by_dynamic_columns(df, y_tolerance=2, x_bin_width=20):
        df = df.sort_values(by=["page", "y0", "x0"]).reset_index(drop=True)
        structured_rows = []

        for page in df["page"].unique():
            page_df = df[df["page"] == page].copy()
            rows = []

            for _, row in page_df.iterrows():
                y0 = row["y0"]
                matched = False

                for r in rows:
                    if abs(r["y0"] - y0) <= y_tolerance:
                        r["cells"].append(row)
                        matched = True
                        break

                if not matched:
                    rows.append({"y0": y0, "cells": [row]})

            for r in rows:
                bins = defaultdict(list)
                for cell in r["cells"]:
                    bin_index = int(cell["x0"] // x_bin_width)
                    bins[bin_index].append(cell["text"])

                sorted_bins = [bins[i] for i in sorted(bins)]
                structured_row = [" ".join(bin) for bin in sorted_bins]
                structured_rows.append(structured_row)

        return pd.DataFrame(structured_rows)

    # ---------------- PROCESS ALL PDFs ----------------
    for filename in os.listdir(input_folder):
        if filename.lower().endswith(".pdf"):
            pdf_path = os.path.join(input_folder, filename)

            df = extract_structured_text(pdf_path)
            df2 = group_by_dynamic_columns(df)

            df2["source_pdf"] = filename
            all_dfs.append(df2)

    final_df = pd.concat(all_dfs, ignore_index=True)

    def extract_right_concat(
        df: pd.DataFrame,
        keywords=("Job No", "P.O. #"),
        in_column: str | None = None,
        sep: str = " ",
        case_insensitive: bool = True
    ) -> pd.Series:

        cols = list(df.columns)
        second_last_idx = len(cols) - 2  # index of second-last column

        # Precompute lowercase keywords if needed
        kw_set = {k.lower() if case_insensitive else k for k in keywords}

        def _match(cell) -> bool:
            if cell is None or (isinstance(cell, float) and np.isnan(cell)):
                return False
            s = str(cell)
            s_cmp = s.lower() if case_insensitive else s
            return any(k in s_cmp for k in kw_set)

        def _concat_row(row: pd.Series) -> str | float:
            # Find keyword position
            if in_column is not None:
                if in_column not in df.columns:
                    raise ValueError(f"`in_column` '{in_column}' not in DataFrame columns.")
                pos = cols.index(in_column)
                if not _match(row[in_column]):
                    return np.nan
            else:
                # Search across the entire row for the first occurrence
                pos = None
                for j, c in enumerate(cols):
                    if _match(row[c]):
                        pos = j
                        break
                if pos is None:
                    return np.nan

            # Compute slice start (immediately to the right) and end (second-last col inclusive)
            start = pos + 1
            end = second_last_idx
            if start > end:
                return np.nan  # nothing to extract

            # Extract, drop None/NaN/empty, concatenate
            vals = row.iloc[start:end+1].tolist()
            cleaned = []
            for v in vals:
                if v is None or (isinstance(v, float) and np.isnan(v)):
                    continue
                s = str(v).strip()
                if s:  # exclude empty after trimming
                    cleaned.append(s)
            if not cleaned:
                return np.nan
            return sep.join(cleaned)

        return df.apply(_concat_row, axis=1)
    final_df["DoNotEmail"] = extract_right_concat(final_df, keywords=("Job No", "P.O. #"))

    final_df2 = final_df[['source_pdf', 'DoNotEmail']]
    final_df2 = final_df2[final_df2["DoNotEmail"].str.contains('Pay App|Adj Inv', case=False, na=False)]
    final_df2.drop_duplicates(subset=['source_pdf'], keep='first')

    # Work on a copy with a fresh RangeIndex so iloc works with your window logic
    fd = final_df.reset_index(drop=True)

    keyword = "Email"
    columns_to_check = [0, 1, 2]

    # Find rows where any of these columns contains 'Email' (case-insensitive)
    matches = fd.index[
        fd[columns_to_check]
        .astype(str)
        .apply(lambda row: any(keyword.lower() in str(val).lower() for val in row), axis=1)]

    # Include Email row + next 2 rows
    rows_to_include = []
    for pos in matches:  # pos are now positions 0..len(fd)-1
        end_pos = min(pos + 3, len(fd))
        rows_to_include.extend(range(pos, end_pos))

    rows_to_include = sorted(set(rows_to_include))
    df_filtered = fd.iloc[rows_to_include].reset_index(drop=True)

    # Remove unwanted rows
    df_filtered = df_filtered[(df_filtered[0] != 'Customer Phone') & (df_filtered[0] != 'Customer Fax')]
    df_filtered.drop(columns="DoNotEmail", inplace=True)


    def combine_email_per_pdf_blocks(df_filtered: pd.DataFrame) -> pd.DataFrame:
        if 'source_pdf' not in df_filtered.columns:
            raise ValueError("Expected 'source_pdf' column in df_filtered.")
        src_idx = df_filtered.columns.get_loc('source_pdf')
        end_col = src_idx - 1  # last usable column index

        def norm(x):
            if x is None or (isinstance(x, float) and pd.isna(x)):
                return ""
            return str(x).replace("\u00A0", " ").strip()

        search_cols = [0, 1, 2]
        markers = {"email", "none", "nan", ""}

        out_rows = []
        for pdf, grp in df_filtered.groupby('source_pdf', sort=False):
            email_pos = None
            first_match_row = None

            # Find first occurrence of 'Email'
            for ridx, row in grp.iterrows():
                for c in search_cols:
                    if c > end_col or c >= len(row):
                        continue
                    if "email" in norm(row.iloc[c]).lower():
                        email_pos = c
                        first_match_row = ridx
                        break
                if email_pos is not None:
                    break

            if email_pos is None or end_col < 0:
                out_rows.append({"source_pdf": pdf, "Email": ""})
                continue

            pieces = []
            for ridx, row in grp.iterrows():
                start_col = (email_pos + 1) if ridx == first_match_row else 0
                start_col = max(0, min(start_col, end_col))
                for col_idx in range(start_col, end_col + 1):
                    val = norm(row.iloc[col_idx])
                    if not val or val.lower() in markers:
                        continue
                    pieces.append(val)

            combined = "".join(pieces)
            out_rows.append({"source_pdf": pdf, "Email": combined})

        return pd.DataFrame(out_rows, columns=["source_pdf", "Email"])

    df_email_by_pdf = combine_email_per_pdf_blocks(df_filtered)

    # Replace df_filtered for downstream steps
    df_filtered = df_email_by_pdf[['source_pdf', 'Email']]

    df_filtered = df_filtered.groupby('source_pdf')['Email'].agg(lambda x: ''.join(x)).reset_index()
    df_filtered['Email'] = df_filtered['Email'].str.replace(r'\*+', '', regex=True).str.strip()

    # >>> NEW: Targeted move of only specific notes to Comment, and strip them from Email
    def extract_comments_and_clean(email_str):
        """
        Moves ONLY the following to comments if present anywhere in the string:
        - 'US Mail' or 'USMail'
        - 'read receipt' (with or without parentheses)
        - '& PM' or '&PM'
        Returns (cleaned_email, comments_str).
        """
        s = email_str or ""
        comments = []

        # Detect presence (case-insensitive) for standardized labels
        if re.search(r'\bUS\s*Mail\b', s, flags=re.IGNORECASE):
            comments.append("US Mail")
        if re.search(r'\bUSMail\b', s, flags=re.IGNORECASE):
            comments.append("USMail")
        if re.search(r'\bread\s+receipt\b', s, flags=re.IGNORECASE):
            comments.append("read receipt")
        if re.search(r'&\s*PM\b', s):
            comments.append("PM")

        # Remove only those occurrences (including common delimiters around them)
        # Slash before US Mail/USMail (e.g., '/US Mail', '/USMail')
        s = re.sub(r'(?i)\s*/\s*US\s*Mail', '', s)
        s = re.sub(r'(?i)\s*/\s*USMail', '', s)

        # 'read receipt' with or without parentheses
        s = re.sub(r'(?i)\s*\(\s*read\s+receipt\s*\)\s*', '', s)  # (read receipt)
        s = re.sub(r'(?i)\s*read\s+receipt\s*', '', s)            # read receipt

        # '& PM' or '&PM'
        s = re.sub(r'\s*&\s*PM\b', '', s)

        # Tidy spaces and commas after removals
        s = re.sub(r'\s+', ' ', s)
        s = re.sub(r'\s*,\s*', ', ', s)
        s = re.sub(r'(,\s*){2,}', ', ', s)
        s = s.strip(' ,')

        # De-duplicate comments preserving order
        seen = set()
        comments_unique = []
        for c in comments:
            if c not in seen:
                comments_unique.append(c)
                seen.add(c)

        return s, '; '.join(comments_unique)

    # Apply extraction
    clean_cols = df_filtered['Email'].apply(extract_comments_and_clean)
    df_filtered['Email'] = clean_cols.apply(lambda x: x[0])     # cleaned email string
    df_filtered['Comment'] = clean_cols.apply(lambda x: x[1])   # ONLY targeted notes
    df_filtered['Email'] = df_filtered['Email'].str.replace(r'(?i)/\s*cc\s*([A-Za-z0-9._%+-]+)', r';\1', regex=True)
    df_filtered['Email'] = df_filtered['Email'].str.replace(r'(?i)/\s*cc\b', r';', regex=True)
    df_filtered['Email'] = df_filtered['Email'].str.replace(r'\s*/cc\s*', ';', flags=re.IGNORECASE, regex=True)
    df_filtered['Email'] = df_filtered['Email'].str.replace(r'/', ';', regex=True)


    def normalize_domains(email_str):
        """
        Normalize ';' separated tokens by using the domain from the FIRST email:
        - If a token contains '@', replace everything from '@' onward with the first domain.
        - If a token has no '@' but is a valid local-part (letters/digits/._%+-), append the first domain.
        - Non-email tokens are left untouched.
        Also normalizes commas to semicolons and tidies separators.
        """
        if not isinstance(email_str, str) or not email_str.strip():
            return email_str

        # Unify commas to semicolons and trim
        s = re.sub(r'\s*,\s*', ';', email_str.strip())

        # Split on semicolons
        parts = [p.strip() for p in s.split(';') if p.strip()]
        if not parts:
            return email_str

        # Find domain from the first email-like token that contains '@'
        domain = None
        for p in parts:
            if '@' in p:
                at_idx = p.find('@')
                dom = p[at_idx:]  # includes '@'
                if len(dom) > 1:
                    domain = dom
                    break

        # If no domain found, do nothing
        if not domain:
            out = ';'.join(parts)
            out = re.sub(r'\s*;\s*', ';', out)
            out = re.sub(r';{2,}', ';', out)
            return out.strip(' ;,')

        normalized = []
        for i, p in enumerate(parts):
            if i == 0:
                # Keep the first token as-is (it carries the canonical domain)
                normalized.append(p)
                continue

            token = p.strip(' ,')

            if '@' in token:
                # Replace everything from '@' with the first domain
                local = token.split('@', 1)[0]
                normalized.append(local + domain)
            else:
                # Append domain if token is a valid local-part (no '@' present)
                # Accept letters/digits and . _ % + - ; no need to require a dot
                if re.match(r'^[A-Za-z0-9._%+-]+$', token):
                    normalized.append(token + domain)
                else:
                    normalized.append(token)  # leave non-email tokens

        out = ';'.join(normalized)
        out = re.sub(r'\s*;\s*', ';', out)
        out = re.sub(r';{2,}', ';', out)
        return out.strip(' ;,')


    # >>> NEW: Normalize domains across ';' separated tokens using the first email's domain
    df_filtered['Email'] = df_filtered['Email'].apply(normalize_domains)
    df_filtered = df_filtered.merge(final_df2, on='source_pdf', how='left')


    def dedupe_semicolon_list(s):
        if not isinstance(s, str) or not s.strip():
            return s
        parts = [p.strip() for p in s.split(';') if p.strip()]
        seen = set()
        out = []
        for p in parts:
            if p not in seen:
                out.append(p)
                seen.add(p)
        return ';'.join(out)

    df_filtered['Email'] = df_filtered['Email'].apply(dedupe_semicolon_list)

    # Ensure folders exist (if not already created above)
    os.makedirs(base_output_folder, exist_ok=True)
    os.makedirs(no_email_folder, exist_ok=True)
    os.makedirs(do_not_email_folder, exist_ok=True)  # <<< make sure this exists

    # ---------------- GROUP PDFs BY EMAIL (with DoNotEmail precedence) ----------------
    for _, row in df_filtered.iterrows():
        pdf_name = row['source_pdf']
        src_pdf_path = os.path.join(input_folder, pdf_name)

        # 1) >>> PRIORITY RULE: If DoNotEmail is present (non-NaN/non-empty), put PDF in DoNotEmail folder and skip further logic
        dni_val = row.get('DoNotEmail', None)
        dni_has_value = isinstance(dni_val, str) and dni_val.strip() != ""
        if (dni_val is not None) and (not isinstance(dni_val, float)) and dni_has_value:
            dest_pdf_path = os.path.join(do_not_email_folder, pdf_name)
            if not os.path.exists(dest_pdf_path):  # Avoid duplicate copies
                shutil.copy(src_pdf_path, dest_pdf_path)
            # Skip to next PDF; DoNotEmail takes precedence over NoEmailFound and email grouping
            continue

        # 2) Existing behavior: split Email tokens and determine grouping
        # Normalize split on semicolon OR comma and drop empties
        email_raw = row.get('Email', "")
        emails = [e.strip() for e in re.split(r'[;,]', str(email_raw)) if e.strip()]

        # If NO '@' appears anywhere in the cleaned Email string, move to NoEmailFound (keep original filename)
        emails_with_at = [e for e in emails if '@' in e]
        if not emails_with_at:
            dest_pdf_path = os.path.join(no_email_folder, pdf_name)
            if not os.path.exists(dest_pdf_path):  # Avoid duplicate copies
                shutil.copy(src_pdf_path, dest_pdf_path)
            continue

        # Use cleaned emails (with '@') for folder grouping
        for email in emails_with_at:
            email_folder = os.path.join(base_output_folder, email)
            os.makedirs(email_folder, exist_ok=True)

            dest_pdf_path = os.path.join(email_folder, pdf_name)
            if not os.path.exists(dest_pdf_path):  # Avoid duplicate copies
                shutil.copy(src_pdf_path, dest_pdf_path)

    # Write mapping
    excel_path = os.path.join(base_output_folder, "Email_and_PDF_Mapping.xlsx")
    df_filtered_sorted = df_filtered.sort_values(by="Email")
    df_filtered_sorted.to_excel(excel_path, index=False)

    print("✅ PDFs have been grouped by email into folders inside:", base_output_folder)
    print(f"🚫 PDFs flagged with DoNotEmail were placed in: {do_not_email_folder}")
    print(f"📂 PDFs with blank Email were placed in: {no_email_folder}")



def ask_to_run_again() -> bool:
    try:
        while True:
            resp = input("\nDo you have more grouping tasks to run for other split PDF invoices? (yes/no): ").strip().lower()
            if resp in {"y", "yes"}:
                return True
            elif resp in {"n", "no"}:
                return False
            else:
                print("Please answer 'yes' or 'no'.")
    except (EOFError, KeyboardInterrupt):
        # If input is not available (e.g., piped execution), default to terminating
        print("\nNo input detected. Exiting.")
        return False


if __name__ == "__main__":
    # Global loop + error handling
    while True:
        try:
            main()
        except Exception as e:
            # Print a clear error message and full stack trace before closing
            print("\n❌ An unexpected error occurred:")
            print(f"   {type(e).__name__}: {e}")
            print("\n--- Stack Trace ---")
            traceback.print_exc()
            print("-------------------\n")
            # Program closes after showing error (per your request)
            sys.exit(1)

        # Ask the user if they want to run again
        if not ask_to_run_again():
            print("👋 Done. Exiting.")
            sys.exit(0)
        # Otherwise, loop continues and `main()` runs anew



✅ PDFs have been grouped by email into folders inside: C:\Users\IlaBarshilia\Downloads\Grouping PDFs by Email\Grouped_PDFs
🚫 PDFs flagged with DoNotEmail were placed in: C:\Users\IlaBarshilia\Downloads\Grouping PDFs by Email\Grouped_PDFs\DoNotEmail
📂 PDFs with blank Email were placed in: C:\Users\IlaBarshilia\Downloads\Grouping PDFs by Email\Grouped_PDFs\NoEmailFound
👋 Done. Exiting.


SystemExit: 0

C:\Users\IlaBarshilia\AppData\Roaming\Python\Python311\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [5]:

import sys
import traceback

def main():
    import os
    import pandas as pd
    import fitz
    from collections import defaultdict
    import shutil
    import re
    import numpy as np

    # Configuration
    input_folder = input("Enter the path to where your PDF files are kept that you want to email to customers: ")
    base_output_folder = os.path.join(input_folder, "Grouped_PDFs")
    os.makedirs(base_output_folder, exist_ok=True)

    # Special folders
    no_email_folder = os.path.join(base_output_folder, "NoEmailFound")
    os.makedirs(no_email_folder, exist_ok=True)

    do_not_email_folder = os.path.join(base_output_folder, "DoNotEmail")
    os.makedirs(do_not_email_folder, exist_ok=True)

    # Store all PDFs' structured data
    all_dfs = []

    def extract_structured_text(pdf_path):
        doc = fitz.open(pdf_path)
        all_blocks = []

        for page_num, page in enumerate(doc):
            blocks = page.get_text("dict")["blocks"]
            for b in blocks:
                if "lines" in b:
                    for line in b["lines"]:
                        for span in line["spans"]:
                            bbox = tuple(round(coord) for coord in span["bbox"])
                            all_blocks.append({
                                "page": page_num + 1,
                                "text": span["text"],
                                "x0": bbox[0],
                                "y0": bbox[1],
                                "x1": bbox[2],
                                "y1": bbox[3]
                            })

        df = pd.DataFrame(all_blocks)
        return df.sort_values(by=["page", "y0", "x0"])

    def group_by_dynamic_columns(df, y_tolerance=2, x_bin_width=20):
        df = df.sort_values(by=["page", "y0", "x0"]).reset_index(drop=True)
        structured_rows = []

        for page in df["page"].unique():
            page_df = df[df["page"] == page].copy()
            rows = []

            for _, row in page_df.iterrows():
                y0 = row["y0"]
                matched = False

                for r in rows:
                    if abs(r["y0"] - y0) <= y_tolerance:
                        r["cells"].append(row)
                        matched = True
                        break

                if not matched:
                    rows.append({"y0": y0, "cells": [row]})

            for r in rows:
                bins = defaultdict(list)
                for cell in r["cells"]:
                    bin_index = int(cell["x0"] // x_bin_width)
                    bins[bin_index].append(cell["text"])

                sorted_bins = [bins[i] for i in sorted(bins)]
                structured_row = [" ".join(bin) for bin in sorted_bins]
                structured_rows.append(structured_row)

        return pd.DataFrame(structured_rows)

    # ---------------- PROCESS ALL PDFs ----------------
    for filename in os.listdir(input_folder):
        if filename.lower().endswith(".pdf"):
            pdf_path = os.path.join(input_folder, filename)

            df = extract_structured_text(pdf_path)
            df2 = group_by_dynamic_columns(df)

            df2["source_pdf"] = filename
            all_dfs.append(df2)

    final_df = pd.concat(all_dfs, ignore_index=True)

    # --- Extract right concat helper
    def extract_right_concat(
        df: pd.DataFrame,
        keywords=("Job No", "P.O. #"),
        in_column: str | None = None,
        sep: str = " ",
        case_insensitive: bool = True
    ) -> pd.Series:

        cols = list(df.columns)
        second_last_idx = len(cols) - 2  # index of second-last column

        kw_set = {k.lower() if case_insensitive else k for k in keywords}

        def _match(cell) -> bool:
            if cell is None or (isinstance(cell, float) and np.isnan(cell)):
                return False
            s = str(cell)
            s_cmp = s.lower() if case_insensitive else s
            return any(k in s_cmp for k in kw_set)

        def _concat_row(row: pd.Series) -> str | float:
            # Find keyword position
            if in_column is not None:
                if in_column not in df.columns:
                    raise ValueError(f"`in_column` '{in_column}' not in DataFrame columns.")
                pos = cols.index(in_column)
                if not _match(row[in_column]):
                    return np.nan
            else:
                # Search across the entire row for the first occurrence
                pos = None
                for j, c in enumerate(cols):
                    if _match(row[c]):
                        pos = j
                        break
                if pos is None:
                    return np.nan

            # Compute slice start (immediately to the right) and end (second-last col inclusive)
            start = pos + 1
            end = second_last_idx
            if start > end:
                return np.nan  # nothing to extract

            # Extract, drop None/NaN/empty, concatenate
            vals = row.iloc[start:end+1].tolist()
            cleaned = []
            for v in vals:
                if v is None or (isinstance(v, float) and np.isnan(v)):
                    continue
                s = str(v).strip()
                if s:
                    cleaned.append(s)
            if not cleaned:
                return np.nan
            return sep.join(cleaned)

        return df.apply(_concat_row, axis=1)

    final_df["DoNotEmail"] = extract_right_concat(final_df, keywords=("Job No", "P.O. #"))

    final_df2 = final_df[['source_pdf', 'DoNotEmail']]
    final_df2 = final_df2[final_df2["DoNotEmail"].str.contains('Pay App|Adj Inv', case=False, na=False)]
    # ✅ assign drop_duplicates result
    final_df2 = final_df2.drop_duplicates(subset=['source_pdf'], keep='first')

    # Work on a copy with a fresh RangeIndex so iloc works with your window logic
    fd = final_df.reset_index(drop=True)

    keyword = "Email"
    columns_to_check = [0, 1, 2]

    # Find rows where any of these columns contains 'Email' (case-insensitive)
    matches = fd.index[
        fd[columns_to_check]
        .astype(str)
        .apply(lambda row: any(keyword.lower() in str(val).lower() for val in row), axis=1)
    ]

    # Include Email row + next 2 rows
    rows_to_include = []
    for pos in matches:
        end_pos = min(pos + 3, len(fd))
        rows_to_include.extend(range(pos, end_pos))

    rows_to_include = sorted(set(rows_to_include))
    df_filtered = fd.iloc[rows_to_include].reset_index(drop=True)

    # Remove unwanted rows
    df_filtered[0] = df_filtered[0].astype(str).str.replace("\u00A0", " ").str.strip()
    df_filtered = df_filtered[(df_filtered[0] != 'Customer Phone') & (df_filtered[0] != 'Customer Fax')]
    df_filtered.drop(columns="DoNotEmail", inplace=True)

    # --- Combine email per PDF blocks
    def combine_email_per_pdf_blocks(df_filtered: pd.DataFrame) -> pd.DataFrame:
        if 'source_pdf' not in df_filtered.columns:
            raise ValueError("Expected 'source_pdf' column in df_filtered.")
        src_idx = df_filtered.columns.get_loc('source_pdf')
        end_col = src_idx - 1  # last usable column index

        def norm(x):
            if x is None or (isinstance(x, float) and pd.isna(x)):
                return ""
            return str(x).replace("\u00A0", " ").strip()

        search_cols = [0, 1, 2]
        markers = {"email", "none", "nan", ""}

        out_rows = []
        for pdf, grp in df_filtered.groupby('source_pdf', sort=False):
            email_pos = None
            first_match_row = None

            # Find first occurrence of 'Email'
            for ridx, row in grp.iterrows():
                for c in search_cols:
                    if c > end_col or c >= len(row):
                        continue
                    if "email" in norm(row.iloc[c]).lower():
                        email_pos = c
                        first_match_row = ridx
                        break
                if email_pos is not None:
                    break

            if email_pos is None or end_col < 0:
                out_rows.append({"source_pdf": pdf, "Email": ""})
                continue

            pieces = []
            for ridx, row in grp.iterrows():
                start_col = (email_pos + 1) if ridx == first_match_row else 0
                start_col = max(0, min(start_col, end_col))
                for col_idx in range(start_col, end_col + 1):
                    val = norm(row.iloc[col_idx])
                    if not val or val.lower() in markers:
                        continue
                    pieces.append(val)

            # 🔁 REVERTED: join with NO space to avoid breaking emails across spans
            combined = "".join(pieces)
            out_rows.append({"source_pdf": pdf, "Email": combined})

        return pd.DataFrame(out_rows, columns=["source_pdf", "Email"])

    df_email_by_pdf = combine_email_per_pdf_blocks(df_filtered)

    # Replace df_filtered for downstream steps
    df_filtered = df_email_by_pdf[['source_pdf', 'Email']]

    df_filtered = df_filtered.groupby('source_pdf')['Email'].agg(lambda x: ''.join(x)).reset_index()
    df_filtered['Email'] = df_filtered['Email'].str.replace(r'\*+', '', regex=True).str.strip()

    # --- Extract targeted comments and clean email field
    def extract_comments_and_clean(email_str):
        s = email_str or ""
        comments = []

        # Detect presence (case-insensitive) for standardized labels
        if re.search(r'\bUS\s*Mail\b', s, flags=re.IGNORECASE):
            comments.append("US Mail")
        if re.search(r'\bUSMail\b', s, flags=re.IGNORECASE):
            comments.append("USMail")
        if re.search(r'\bread\s+receipt\b', s, flags=re.IGNORECASE):
            comments.append("read receipt")

        # Detect & PM (HTML-encoded or plain)
        if re.search(r'(?i)(?:&amp;|\&)\s*PM\b', s):
            comments.append("PM")

        # Remove only those occurrences
        s = re.sub(r'(?i)\s*/\s*US\s*Mail', '', s)
        s = re.sub(r'(?i)\s*/\s*USMail', '', s)
        s = re.sub(r'(?i)\s*\(\s*read\s+receipt\s*\)\s*', '', s)
        s = re.sub(r'(?i)\s*read\s+receipt\s*', '', s)
        s = re.sub(r'(?i)\s*&amp;\s*PM\b', '', s)
        s = re.sub(r'(?i)\s*&\s*PM\b', '', s)
        s = re.sub(r'(?i)\s*&PM\b', '', s)

        # Tidy spaces and commas after removals
        s = re.sub(r'\s+', ' ', s)
        s = re.sub(r'\s*,\s*', ', ', s)
        s = re.sub(r'(,\s*){2,}', ', ', s)
        s = s.strip(' ,')

        # De-duplicate comments preserving order
        seen = set()
        comments_unique = []
        for c in comments:
            if c not in seen:
                comments_unique.append(c)
                seen.add(c)

        return s, '; '.join(comments_unique)

    # Apply extraction
    clean_cols = df_filtered['Email'].apply(extract_comments_and_clean)
    df_filtered['Email'] = clean_cols.apply(lambda x: x[0])     # cleaned email string
    df_filtered['Comment'] = clean_cols.apply(lambda x: x[1])   # ONLY targeted notes

    # Normalize "cc" separators and slashes to semicolons
    df_filtered['Email'] = df_filtered['Email'].str.replace(r'(?i)/\s*cc\s*([A-Za-z0-9._%+-]+)', r';\1', regex=True)
    df_filtered['Email'] = df_filtered['Email'].str.replace(r'(?i)/\s*cc\b', r';', regex=True)
    df_filtered['Email'] = df_filtered['Email'].str.replace(r'\s*/cc\s*', ';', flags=re.IGNORECASE, regex=True)
    df_filtered['Email'] = df_filtered['Email'].str.replace(r'/', ';', regex=True)

    # --- Normalize domains across ';' separated tokens using the first email's domain
    def normalize_domains(email_str):
        if not isinstance(email_str, str) or not email_str.strip():
            return email_str

        # Unify commas to semicolons and trim
        s = re.sub(r'\s*,\s*', ';', email_str.strip())

        # Split on semicolons
        parts = [p.strip() for p in s.split(';') if p.strip()]
        if not parts:
            return email_str

        # Find domain from the first email-like token that contains '@'
        domain = None
        for p in parts:
            if '@' in p:
                at_idx = p.find('@')
                dom = p[at_idx:]  # includes '@'
                if len(dom) > 1:
                    domain = dom
                    break

        # If no domain found, do nothing
        if not domain:
            out = ';'.join(parts)
            out = re.sub(r'\s*;\s*', ';', out)
            out = re.sub(r';{2,}', ';', out)
            return out.strip(' ;,')

        normalized = []
        for i, p in enumerate(parts):
            if i == 0:
                normalized.append(p)
                continue

            token = p.strip(' ,')

            if '@' in token:
                local = token.split('@', 1)[0]
                normalized.append(local + domain)
            else:
                if re.match(r'^[A-Za-z0-9._%+-]+$', token):
                    normalized.append(token + domain)
                else:
                    normalized.append(token)

        out = ';'.join(normalized)
        out = re.sub(r'\s*;\s*', ';', out)
        out = re.sub(r';{2,}', ';', out)
        return out.strip(' ;,')

    df_filtered['Email'] = df_filtered['Email'].apply(normalize_domains)
    df_filtered = df_filtered.merge(final_df2, on='source_pdf', how='left')

    # Deduplicate email tokens joined by ';'
    def dedupe_semicolon_list(s):
        if not isinstance(s, str) or not s.strip():
            return s
        parts = [p.strip() for p in s.split(';') if p.strip()]
        seen = set()
        out = []
        for p in parts:
            if p not in seen:
                out.append(p)
                seen.add(p)
        return ';'.join(out)

    df_filtered['Email'] = df_filtered['Email'].apply(dedupe_semicolon_list)

    # Ensure folders exist
    os.makedirs(base_output_folder, exist_ok=True)
    os.makedirs(no_email_folder, exist_ok=True)
    os.makedirs(do_not_email_folder, exist_ok=True)

    # ---------------- GROUP PDFs BY COMPOSITE EMAIL FOLDER ----------------
    # Folder-name sanitizer (keeps '@', '.', and ';'; replaces filesystem-illegal chars)
    def safe_folder_name(name: str) -> str:
        if not isinstance(name, str):
            name = str(name or "")
        # Replace illegal path characters on Windows and most OSes: <>:"/\|?*
        name = re.sub(r'[<>:"/\\|?*]', '_', name)
        # Collapse whitespace and trim trailing dots/spaces
        name = re.sub(r'\s+', ' ', name).strip(' .')
        # Limit very long names (Windows NTFS ~255 chars; keep headroom)
        return name[:240] if len(name) > 240 else name

    # Extract a deterministic composite set of emails for a row
    def extract_email_set(email_field: str) -> list[str]:
        """
        Splits Email on ';' or ',' and returns a deduped, sorted list of
        tokens that look like real email addresses. Sorting ensures PDFs
        with the same email set map to the same folder name regardless of order.
        """
        s = (email_field or "").strip()
        if not s:
            return []

        tokens = [t.strip() for t in re.split(r'[;,]', s) if t.strip()]

        # Basic email pattern with a TLD; tolerates subdomains
        email_regex = re.compile(r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$')

        # Salvage first email-like substring if token doesn't match exactly
        email_finder = re.compile(r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}')

        found = []
        for t in tokens:
            if email_regex.match(t):
                found.append(t)
            else:
                m = email_finder.search(t)
                if m:
                    found.append(m.group(0))

        # Deduplicate preserving order
        seen = set()
        deduped = []
        for e in found:
            if e not in seen:
                deduped.append(e)
                seen.add(e)

        # Sort case-insensitively to ensure stable composite folder names
        deduped.sort(key=lambda x: (x.lower(), x))
        return deduped

    # Build composite folder column for mapping/debug
    df_filtered['FinalFolderName'] = df_filtered['Email'].apply(
        lambda s: ';'.join(extract_email_set(s)) if s and extract_email_set(s) else ''
    )
    df_filtered['Email Verified? (Yes/No)'] = ""

    for _, row in df_filtered.iterrows():
        pdf_name = row['source_pdf']
        src_pdf_path = os.path.join(input_folder, pdf_name)

        # PRIORITY RULE: If DoNotEmail has value, copy to DoNotEmail and skip
        dni_val = row.get('DoNotEmail', None)
        if isinstance(dni_val, str) and dni_val.strip():
            dest_pdf_path = os.path.join(do_not_email_folder, pdf_name)
            if not os.path.exists(dest_pdf_path):
                shutil.copy(src_pdf_path, dest_pdf_path)
            continue

        composite = (row.get('FinalFolderName', '') or '').strip()
        if not composite:
            # No valid emails -> NoEmailFound
            dest_pdf_path = os.path.join(no_email_folder, pdf_name)
            if not os.path.exists(dest_pdf_path):
                shutil.copy(src_pdf_path, dest_pdf_path)
            continue

        # Create the single composite folder (exact list of emails joined by ';')
        folder_name = safe_folder_name(composite)
        email_folder = os.path.join(base_output_folder, folder_name)
        os.makedirs(email_folder, exist_ok=True)

        dest_pdf_path = os.path.join(email_folder, pdf_name)
        if not os.path.exists(dest_pdf_path):
            shutil.copy(src_pdf_path, dest_pdf_path)

    # Write mapping (include composite folder for easy cross-check)
    excel_path = os.path.join(base_output_folder, "Email_and_PDF_Mapping.xlsx")
    df_filtered_sorted = df_filtered.sort_values(by="FinalFolderName")
    df_filtered_sorted.to_excel(excel_path, index=False)

    print("✅ PDFs have been grouped by composite email into folders inside:", base_output_folder)
    print(f"🚫 PDFs flagged with DoNotEmail were placed in: {do_not_email_folder}")
    print(f"📂 PDFs with blank/invalid Email were placed in: {no_email_folder}")
    print(f"📒 Mapping written to: {excel_path}")


def ask_to_run_again() -> bool:
    try:
        while True:
            resp = input("\nDo you have more grouping tasks to run for other split PDF invoices? (yes/no): ").strip().lower()
            if resp in {"y", "yes"}:
                return True
            elif resp in {"n", "no"}:
                return False
            else:
                print("Please answer 'yes' or 'no'.")
    except (EOFError, KeyboardInterrupt):
        # If input is not available (e.g., piped execution), default to terminating
        print("\nNo input detected. Exiting.")
        return False


def wait_before_exit_prompt():
    try:
        input("⏸ Press Enter to exit...")
    except (EOFError, KeyboardInterrupt):
        # Non-interactive environment or user interrupted; just proceed to exit
        pass


if __name__ == "__main__":
    # Global loop + error handling
    while True:
        try:
            main()
        except Exception as e:
            # Print a clear error message and full stack trace before closing
            print("\n❌ An unexpected error occurred:")
            print(f"   {type(e).__name__}: {e}")
            print("\n--- Stack Trace ---")
            traceback.print_exc()
            print("-------------------\n")

            # Pause so you can capture the error
            wait_before_exit_prompt()

            # Program closes after showing error
            sys.exit(1)

        # Ask the user if they want to run again
        if not ask_to_run_again():
            print("👋 Done. Exiting.")
            sys.exit(0)
        # Otherwise, loop continues and `main()` runs anew


✅ PDFs have been grouped by composite email into folders inside: C:\Users\IlaBarshilia\Downloads\split_invoices_of_JAX Oct CH Billing - Pt 2_2025-Nov-05\Grouped_PDFs
🚫 PDFs flagged with DoNotEmail were placed in: C:\Users\IlaBarshilia\Downloads\split_invoices_of_JAX Oct CH Billing - Pt 2_2025-Nov-05\Grouped_PDFs\DoNotEmail
📂 PDFs with blank/invalid Email were placed in: C:\Users\IlaBarshilia\Downloads\split_invoices_of_JAX Oct CH Billing - Pt 2_2025-Nov-05\Grouped_PDFs\NoEmailFound
📒 Mapping written to: C:\Users\IlaBarshilia\Downloads\split_invoices_of_JAX Oct CH Billing - Pt 2_2025-Nov-05\Grouped_PDFs\Email_and_PDF_Mapping.xlsx
👋 Done. Exiting.


SystemExit: 0

C:\Users\IlaBarshilia\AppData\Roaming\Python\Python311\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [12]:
#!/usr/bin/env python3
"""
Send grouped PDFs via email based on a mapping Excel.

Workflow
--------
1) Ask user for the *same base input folder* that contains the PDFs and the `Grouped_PDFs` subfolder.
2) Load `Grouped_PDFs/Email_and_PDF_Mapping.xlsx`.
3) Filter rows where:
   - `Email Verified? (Yes/No)` == 'Yes'
   - and `FinalFolderName` (or fallback to `CompositeEmailFolder`) is not blank.
4) For each such row, gather all PDFs inside that folder and compose an email.
5) Default mode: create an .eml draft per folder (no credentials required).
   Optional: --send to actually send via SMTP using OAuth2 or app password.

Notes
-----
- Draft mode produces portable .eml files in `Grouped_PDFs/OutgoingDrafts` you can open/send via Outlook.
- SMTP send mode requires valid credentials and modern auth (OAuth2). Basic auth is deprecated for M365.
- Subject/Body can be supplied per-row via columns `Subject` and `Body`. If absent, defaults are used.
- Supports multiple recipients separated by ';' or ','.

"""

import os
import sys
import re
import smtplib
import ssl
from pathlib import Path
from typing import List, Optional
import pandas as pd
from email.message import EmailMessage
from datetime import datetime

# --- Constants ---
MAPPING_FILENAME = "Email_and_PDF_Mapping.xlsx"
GROUPED_FOLDER_NAME = "Grouped_PDFs"
OUTGOING_DRAFTS_DIR = "OutgoingDrafts"
LOG_FILENAME = "SendLog.csv"

# Default templates (used if mapping doesn't have Subject/Body columns)
DEFAULT_SUBJECT = "Your documents from {folder_name} ({pdf_count} attachment(s))"
DEFAULT_BODY = (
    "Hi,\n\n"
    "Please find attached the documents for your records.\n"
    "Folder: {folder_name}\n"
    "Attachments: {pdf_count}\n\n"
    "Regards,\n"
    "{sender_name}"
)

# Simple email validation
EMAIL_RE = re.compile(r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$")


def is_valid_email(s: str) -> bool:
    return bool(s and EMAIL_RE.match(s.strip()))


def split_emails(s: str) -> List[str]:
    if not isinstance(s, str):
        return []
    tokens = [t.strip() for t in re.split(r"[;,]", s) if t.strip()]
    # Dedupe while preserving order and keep only valid
    seen = set()
    out = []
    for t in tokens:
        # salvage email substring if needed
        m = re.search(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}", t)
        if m:
            e = m.group(0)
            if e not in seen:
                out.append(e)
                seen.add(e)
    return out


def safe_path_component(name: str) -> str:
    if not isinstance(name, str):
        name = str(name or "")
    # Replace illegal path chars but keep '@', '.', ';'
    name = re.sub(r'[<>:"/\\|?*]', '_', name)
    name = re.sub(r"\s+", " ", name).strip(" .")
    return name[:240] if len(name) > 240 else name


def build_message(sender: str, recipients: List[str], subject: str, body: str,
                  attachments: List[Path]) -> EmailMessage:
    msg = EmailMessage()
    msg["From"] = sender
    msg["To"] = ", ".join(recipients)
    msg["Subject"] = subject
    msg.set_content(body)

    for ap in attachments:
        # Try to guess subtype
        subtype = "pdf" if ap.suffix.lower() == ".pdf" else None
        with open(ap, "rb") as f:
            data = f.read()
        if subtype == "pdf":
            msg.add_attachment(data, maintype="application", subtype="pdf", filename=ap.name)
        else:
            # generic binary
            msg.add_attachment(data, maintype="application", subtype="octet-stream", filename=ap.name)
    return msg


def save_eml(msg: EmailMessage, dest_dir: Path, base_name: str) -> Path:
    dest_dir.mkdir(parents=True, exist_ok=True)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    fn = safe_path_component(f"{base_name}_{ts}.eml")
    eml_path = dest_dir / fn
    with open(eml_path, "wb") as f:
        f.write(msg.as_bytes())
    return eml_path


def send_smtp(msg: EmailMessage, smtp_server: str, smtp_port: int,
              username: str, password: Optional[str] = None, use_tls: bool = True) -> None:
    """
    Basic SMTP send. NOTE: For Microsoft 365, basic auth is deprecated.
    Prefer OAuth2 or app passwords. This function is provided for environments
    where SMTP AUTH is enabled.
    """
    if use_tls:
        context = ssl.create_default_context()
        with smtplib.SMTP(smtp_server, smtp_port) as server:
            server.starttls(context=context)
            server.login(username, password or "")
            server.send_message(msg)
    else:
        with smtplib.SMTP_SSL(smtp_server, smtp_port) as server:
            server.login(username, password or "")
            server.send_message(msg)


def main():
    base_input = input("Enter the SAME base input folder used for grouping (contains Grouped_PDFs): ").strip()
    if not base_input:
        print("No folder provided. Exiting.")
        sys.exit(1)

    base_path = Path(base_input)
    grouped_path = base_path / GROUPED_FOLDER_NAME
    mapping_path = grouped_path / MAPPING_FILENAME

    if not grouped_path.exists():
        print(f"❌ Missing '{GROUPED_FOLDER_NAME}' under: {base_path}")
        sys.exit(1)
    if not mapping_path.exists():
        print(f"❌ Mapping file not found: {mapping_path}")
        sys.exit(1)

    # Load mapping
    try:
        df = pd.read_excel(mapping_path, engine="openpyxl")
    except Exception as e:
        print(f"❌ Failed to read mapping Excel: {e}")
        sys.exit(1)

    # Normalize column names
    cols_norm = {c: c.strip() for c in df.columns}
    df.rename(columns=cols_norm, inplace=True)

    # Required columns
    required_cols = ["source_pdf", "Email", "Email Verified? (Yes/No)"]
    for rc in required_cols:
        if rc not in df.columns:
            print(f"❌ Required column missing in mapping: {rc}")
            sys.exit(1)

    # Folder name column (prefer FinalFolderName, else CompositeEmailFolder)
    folder_col = None
    if "FinalFolderName" in df.columns:
        folder_col = "FinalFolderName"
    else:
        print("❌ Neither 'FinalFolderName' nor 'CompositeEmailFolder' present in mapping.")
        sys.exit(1)

    # Optional per-row subject/body
    has_subject = "Subject" in df.columns
    has_body = "Body" in df.columns

    # Mode: drafts or send
    mode = input("Mode: type 'send' to email via SMTP, or press Enter for DRAFTS (.eml files): ").strip().lower()
    do_send = (mode == "send")

    # If sending, collect SMTP settings
    smtp_server = smtp_port = smtp_user = smtp_pass = None
    if do_send:
        smtp_server = input("SMTP server (e.g., smtp.office365.com): ").strip() or "smtp.office365.com"
        smtp_port_str = input("SMTP port (e.g., 587): ").strip() or "587"
        try:
            smtp_port = int(smtp_port_str)
        except ValueError:
            print("❌ Invalid port. Use a number like 587.")
            sys.exit(1)
        smtp_user = input("SMTP username (your email address): ").strip()
        smtp_pass = input("SMTP password/app password (leave blank if using OAuth in your environment): ").strip()
        if not smtp_user:
            print("❌ SMTP username required to send.")
            sys.exit(1)

    # Sender name and address
    sender_email = input("From email address (e.g., you@company.com): ").strip()
    sender_name = input("Sender display name (optional): ").strip() or "Sender"

    # Results log
    log_rows = []

    
    # Filter rows: Verified == 'Yes', folder present, and DoNotEmail blank
    verified_yes = df["Email Verified? (Yes/No)"].astype(str).str.strip().str.lower() == "yes"
    folder_present = df[folder_col].astype(str).str.strip() != ""
    # Treat NaN or empty strings as "blank"
    do_not_email_blank = df["DoNotEmail"].isna() | (df["DoNotEmail"].astype(str).str.strip() == "")

    filt = verified_yes & folder_present & do_not_email_blank
    df_send = df.loc[filt].copy()


    if df_send.empty:
        print("ℹ️ No rows qualified for sending (Verified=Yes and folder name present).")

    for idx, row in df_send.iterrows():
        folder_name_raw = str(row[folder_col]).strip()
        email_raw = str(row.get("Email", "")).strip()
        recipients = split_emails(email_raw)

        if not recipients:
            log_rows.append({"Folder": folder_name_raw, "Status": "Skipped - No valid recipients", "Detail": email_raw})
            continue

        # Build folder path and collect PDFs
        email_folder_path = grouped_path / safe_path_component(folder_name_raw)
        if not email_folder_path.exists():
            log_rows.append({"Folder": folder_name_raw, "Status": "Skipped - Folder not found", "Detail": str(email_folder_path)})
            continue

        pdfs = sorted([p for p in email_folder_path.glob("*.pdf") if p.is_file()])
        if not pdfs:
            log_rows.append({"Folder": folder_name_raw, "Status": "Skipped - No PDFs in folder", "Detail": str(email_folder_path)})
            continue

        # Subject/body per-row or defaults
        subject_tpl = str(row["Subject"]).strip() if has_subject and pd.notna(row["Subject"]) else DEFAULT_SUBJECT
        body_tpl = str(row["Body"]).strip() if has_body and pd.notna(row["Body"]) else DEFAULT_BODY

        subject = subject_tpl.format(folder_name=folder_name_raw, pdf_count=len(pdfs))
        body = body_tpl.format(folder_name=folder_name_raw, pdf_count=len(pdfs), sender_name=sender_name)

        # Build message
        msg = build_message(sender=sender_email, recipients=recipients, subject=subject, body=body, attachments=pdfs)

        try:
            if do_send:
                send_smtp(msg, smtp_server=smtp_server, smtp_port=smtp_port, username=smtp_user, password=smtp_pass)
                log_rows.append({"Folder": folder_name_raw, "Status": "SENT", "Detail": ", ".join(recipients)})
            else:
                drafts_dir = grouped_path / OUTGOING_DRAFTS_DIR / safe_path_component(folder_name_raw)
                eml_path = save_eml(msg, drafts_dir, base_name="draft")
                log_rows.append({"Folder": folder_name_raw, "Status": "DRAFT_CREATED", "Detail": str(eml_path)})
        except Exception as e:
            log_rows.append({"Folder": folder_name_raw, "Status": "ERROR", "Detail": str(e)})

    # Write log
    log_df = pd.DataFrame(log_rows)
    log_path = grouped_path / LOG_FILENAME
    try:
        log_df.to_csv(log_path, index=False)
        print(f"\n📒 Log written: {log_path}")
    except Exception as e:
        print(f"❌ Failed to write log: {e}")

    if not df_send.empty:
        print(f"\n✅ Completed processing {len(df_send)} row(s).")
        if not do_send:
            print(f"Draft emails (.eml) are in: {grouped_path / OUTGOING_DRAFTS_DIR}")


if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        print("\nInterrupted.")



📒 Log written: C:\Users\IlaBarshilia\Downloads\Grouped_Email_Test Folder\Grouped_PDFs\SendLog.csv

✅ Completed processing 16 row(s).
